# Superoperator simulation

Adapted from Qualtran v0.7.0 `simulation/supertensor.ipynb`, commit `bcb17bb25360ed68ce3bc084645db3e929799389` (Google LLC, Apache-2.0). [Locked source](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/simulation/supertensor.ipynb).

Closed systems use statevectors and unitary matrices. Open systems use density matrices and channels. This notebook composes native numerical tensor owners without importing a reference package. Dense open-system tensors are more expensive; prefer pure-state contraction when appropriate. Run from the repository root.

## Density matrices

A density matrix has probabilities on its diagonal and coherences off its diagonal. We first compare the statevector and density matrix of the coherent plus state. Numerical tensors are PyTorch tensors; native integer indices name connections.

In [ ]:
import torch
import flagquantum as fq
from examples.algorithms.superoperator_workflow import superoperator_example, ZMeasurementDilation
from flagquantum.drawer.tensor_network import draw_tensor_network
plus = superoperator_example("plus")
plus.pure()

In [ ]:
plus.channel()

### Incoherent states

Preparing zero, applying H, and measuring Z creates a classical coin. Measurement copies a result into a retained classical record and an environment. Tracing the environment removes coherence; the record remains a logical output.

In [ ]:
coin_flip = superoperator_example("coin")
coin_figure = draw_tensor_network(coin_flip.doubled_nodes())
rho_coin_flip = coin_flip.channel()
display(rho_coin_flip)

## Channel evolution by matrix-vector multiplication

Native contraction combines leaves directly. For clarity we also reshape the channel and density matrix to demonstrate ordinary matrix-vector evolution. No sampling or normalization is introduced by tensor contraction.

### Four-index superoperator tensors

An ordinary one-qubit gate is a two-index matrix. Its channel has four indices, while a prepared density matrix has two.

In [ ]:
super_h = superoperator_example("hadamard").channel()
super_h.shape

In [ ]:
rho_coin_flip.shape

The axis order is `(output-forward, output-backward, input-forward, input-backward)`. A pure density matrix is `psi[:, None] * psi.conj()[None, :]`. Unitary evolution applies U to the forward state and its elementwise conjugate to the backward state; the network connections supply the transpose structure. State preparation has only output-forward/backward axes; effects have only input-forward/backward axes.

### Reshaping

Row-major reshaping preserves these groups. H leaves the incoherent coin unchanged but maps the coherent plus state to zero deterministically.

In [ ]:
# Row-major vectorization uses (output-forward, output-backward, input-forward, input-backward).
(super_h.reshape(4, 4) @ rho_coin_flip.reshape(4)).reshape(2, 2)

In [ ]:
rho_coherent = plus.channel()
(super_h.reshape(4, 4) @ rho_coherent.reshape(4)).reshape(2, 2)

## Native tensor-network inspection

Every ordinary leaf is paired with its complex conjugate. The measured environment couples those two evolutions. Existing native incidence drawing displays every tensor/axis connection, including hyperedges. These figures preserve connectivity, not Quimb colors, spectral coordinates, or the original musical-score layout.

In [ ]:
coin_network_figure = draw_tensor_network(coin_flip.doubled_nodes())

## System and environment

A channel can be represented as unitary evolution on a larger space followed by an environmental trace (Stinespring dilation). For Z measurement, allocate a fresh zero record, apply H, controlled Z, then H to that record. The input q remains entangled with the record. This concrete example reuses the existing unitary and trace owners; it does not implement a new general channel synthesis API.

In [ ]:
meas_graph = superoperator_example("dilation")
meas_figure = draw_tensor_network(meas_graph.nodes)
# q=wire0, fresh zero record=wire1; spectator=wire2.
circuit = fq.Circuit(3, dtype=torch.complex128)
circuit.append(ZMeasurementDilation())
coherent_output = circuit.state()

The QBit-to-CBit Cast is a numerical identity, not a measurement. Both coherent outputs remain visible in the pure tensor graph, even though c is declared classical. The low-level tensor example exposes that mathematical isometry; the physical coherent circuit retains its quantum carrier. Sampling c and clearing consumed physical slots is a separate dynamic operation.

In [ ]:
dilation_network_figure = draw_tensor_network(meas_graph.nodes)
dilation_isometry = meas_graph.pure()
display(dilation_isometry)

### Measurement as an open-system operation

Discard q after the dilation and retain c. The logical signature now consumes one quantum bit and produces one classical bit. Pure simulation must refuse this discard. The dynamic `ZMeasurementDilation(measured=True)` additionally materializes c in classical slot 0, clears its carrier, and resets consumed q; those physical instructions are counted separately from the numerical trace.

In [ ]:
meas2_graph = superoperator_example("measurement")
measurement_figure = draw_tensor_network(meas2_graph.doubled_nodes())

The pure tensor protocol cannot trace an environment. Its explicit refusal is checked below; the diagnostic wording is native.

In [ ]:
try:
    meas2_graph.pure()
except ValueError as error:
    print("ValueError:", error)

Discard sums probabilities, not amplitudes. It therefore identifies the discarded forward and backward indices after conjugate doubling. In density-matrix notation this is a partial trace. The result below is a rank-four channel from input q to retained record c.

In [ ]:
measurement_network_figure = draw_tensor_network(meas2_graph.doubled_nodes())
measurement_channel = meas2_graph.channel()
display(measurement_channel.shape)

The discarded q couples the two evolutions. Arbitrary quantum channels admit a system-environment description, but this notebook certifies only its concrete plus/H/coin/Z-dilation/discard workflows. Existing Kraus and Choi tools remain owned by the native operator layer.

## Ownership and implementation

Simulation owns numerical leaves, conjugate pairing, explicit trace merging, register-axis grouping, and contraction. The example supplies named integer connections and reuses those owners. Drawing consumes the same nodes without modifying them. Circuit IR and the existing measurement/clear primitives own the physical consumer.

A trace is requested only after its producing leaf. Integer labels remain distinct from physical wire numbers. No foreign graph objects, sentinel classes, second executor, or production registry is introduced. The fresh record is an owned output, not zero scratch. The measured physical form returns consumed q and the record carrier to zero; spectator wire 2 is untouched. Primitive counts and upstream analytical models are reported separately in the acceptance document.